<a href="https://colab.research.google.com/github/ShivalikaSudhas/mini-project/blob/liza-fusion/Fusion_Generalization_Only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from google.colab import files
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Please upload the combined CV + OCR feature CSV from your teammates.")
df = pd.read_csv(csv_files[0])

required = {"document_type", "label"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

cv_columns = [c for c in df.columns if c.startswith("cv_")]
ocr_columns = [c for c in df.columns if c.startswith("ocr_")]
if not cv_columns:
    raise ValueError("No CV feature columns found. Expected column names starting with 'cv_'.")
if not ocr_columns:
    raise ValueError("No OCR feature columns found. Expected column names starting with 'ocr_'.")

df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label", "document_type"]).copy()
df["label"] = df["label"].astype(int)
if not set(df["label"].unique()).issubset({0,1}):
    raise ValueError("Labels must be 0 (genuine) or 1 (forged).")
for c in cv_columns + ocr_columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Rows:", len(df))
print("CV feature columns:", cv_columns)
print("OCR feature columns:", ocr_columns)
print("Document types:", sorted(df["document_type"].astype(str).unique()))
display(df.head())
display(df.groupby(["document_type", "label"]).size().rename("count").reset_index())

In [ ]:
fusion_columns = cv_columns + ocr_columns

def build_classifier():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced")
    )

model_configs = [
    ("CV only", cv_columns),
    ("OCR only", ocr_columns),
    ("CV + OCR fusion", fusion_columns)
]
print("Fusion feature count:", len(fusion_columns))

In [ ]:
X_all = df[cv_columns + ocr_columns]
y_all = df["label"]
if y_all.nunique() != 2 or y_all.value_counts().min() < 2:
    raise ValueError("Need at least two examples from each class for a train/test comparison.")

from sklearn.model_selection import train_test_split
train_idx, test_idx = train_test_split(
    df.index, test_size=0.30, random_state=42, stratify=y_all
)
train_df = df.loc[train_idx]
test_df = df.loc[test_idx]

comparison_rows = []
trained_models = {}
for model_name, cols in model_configs:
    model = build_classifier()
    model.fit(train_df[cols], train_df["label"])
    pred = model.predict(test_df[cols])
    prob = model.predict_proba(test_df[cols])[:, 1]
    trained_models[model_name] = model
    row = {
        "Model": model_name,
        "Accuracy": accuracy_score(test_df["label"], pred),
        "Precision": precision_score(test_df["label"], pred, zero_division=0),
        "Recall": recall_score(test_df["label"], pred, zero_division=0),
        "F1-score": f1_score(test_df["label"], pred, zero_division=0),
        "AUROC": roc_auc_score(test_df["label"], prob) if test_df["label"].nunique() == 2 else np.nan
    }
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(3))

In [ ]:
fusion_model = trained_models["CV + OCR fusion"]
fusion_pred = fusion_model.predict(test_df[fusion_columns])
print(classification_report(
    test_df["label"], fusion_pred,
    labels=[0,1], target_names=["Genuine","Forged"], zero_division=0
))
cm = confusion_matrix(test_df["label"], fusion_pred, labels=[0,1])
ConfusionMatrixDisplay(cm, display_labels=["Genuine","Forged"]).plot()
plt.title("CV + OCR Fusion — random test split")
plt.show()

In [ ]:
document_types = sorted(df["document_type"].astype(str).unique())
if len(document_types) < 2:
    raise ValueError("LODTO requires at least two distinct document types.")

lodto_rows = []
for held_out_type in document_types:
    train_part = df[df["document_type"].astype(str) != held_out_type].copy()
    test_part = df[df["document_type"].astype(str) == held_out_type].copy()

    if train_part["label"].nunique() < 2:
        print(f"Skipping {held_out_type}: training set does not contain both classes.")
        continue
    if test_part["label"].nunique() < 2:
        print(f"Skipping {held_out_type}: held-out type does not contain both classes; AUROC unavailable and classification metrics may be incomplete.")

    model = build_classifier()
    model.fit(train_part[fusion_columns], train_part["label"])
    pred = model.predict(test_part[fusion_columns])
    prob = model.predict_proba(test_part[fusion_columns])[:, 1]

    lodto_rows.append({
        "Held-out document type": held_out_type,
        "Test images": len(test_part),
        "Accuracy": accuracy_score(test_part["label"], pred),
        "Precision": precision_score(test_part["label"], pred, zero_division=0),
        "Recall": recall_score(test_part["label"], pred, zero_division=0),
        "F1-score": f1_score(test_part["label"], pred, zero_division=0),
        "AUROC": roc_auc_score(test_part["label"], prob) if test_part["label"].nunique() == 2 else np.nan
    })

lodto_df = pd.DataFrame(lodto_rows)
display(lodto_df.round(3))

In [ ]:
# Compare the fusion model's training performance with its performance on each unseen document type.
# This is a simple descriptive gap; report the exact calculation in your project write-up.
if not lodto_df.empty:
    train_fusion = build_classifier()
    train_fusion.fit(df[fusion_columns], df["label"])
    train_pred = train_fusion.predict(df[fusion_columns])
    train_f1 = f1_score(df["label"], train_pred, zero_division=0)

    lodto_df["Generalization gap (train F1 - unseen F1)"] = train_f1 - lodto_df["F1-score"]
    print("In-sample training F1 (descriptive only):", round(train_f1, 3))
    display(lodto_df.round(3))
else:
    print("No valid LODTO folds were produced. Check the data and class counts.")

In [ ]:
output_dir = "fusion_generalization_results"
import os
os.makedirs(output_dir, exist_ok=True)

# Save a final fusion classifier trained on all available rows for later integration.
final_fusion_model = build_classifier()
final_fusion_model.fit(df[fusion_columns], df["label"])
joblib.dump(final_fusion_model, os.path.join(output_dir, "fusion_classifier.joblib"))

comparison_df.to_csv(os.path.join(output_dir, "model_comparison.csv"), index=False)
if not lodto_df.empty:
    lodto_df.to_csv(os.path.join(output_dir, "lodto_results.csv"), index=False)

with open(os.path.join(output_dir, "feature_columns.txt"), "w") as f:
    f.write("CV columns:\n" + "\n".join(cv_columns))
    f.write("\n\nOCR columns:\n" + "\n".join(ocr_columns))

print("Saved model and result files in:", output_dir)

In [ ]:
!zip -qr fusion_generalization_results.zip fusion_generalization_results
files.download("fusion_generalization_results.zip")

## Integration handoff checklist

When teammates provide their model outputs, confirm:
- Each row refers to the same source image in both streams.
- CV feature columns start with `cv_`.
- OCR feature columns start with `ocr_`.
- `label` uses 0 = genuine and 1 = forged.
- `document_type` is the actual template/type family.
- No image from a held-out document type appears in training.
- Feature extraction and preprocessing are consistent between training and testing.

This package contains your assigned fusion and generalization code only. It does not include dataset creation, CV model development, or OCR model development.